# Walkthrough d'entrainement ViT sur CIFAR-10

| | |
|---|---|
| **Architecture** | Vision Transformer (ViT) |
| **Sujet** | Pipeline complet de l'image brute a l'evaluation du modele |
| **Papier** | [Dosovitskiy et al. 2020 (*An Image is Worth 16x16 Words*)](https://arxiv.org/abs/2010.11929) |
| **Dataset** | CIFAR-10 (Krizhevsky, 2009) |
| **Etapes couvertes** | Normalisation · augmentations · DataLoader · patches · tokens · entrainement · evaluation · attention |
| **Langage** | Python / PyTorch |

---

Ce notebook suit la meme logique qu'un vrai entrainement ViT, mais avec des lots et des epoques volontairement petits pour rester lisible. L'objectif est de comprendre ce que fait chaque brique et pourquoi elle existe, pas d'atteindre des resultats de pointe.

Le parcours suit cet ordre : normalisation et augmentations → chargement du dataset → sous-echantillonnage → DataLoader → inspection d'un batch → extraction des patches → sequence de tokens → entrainement → evaluation → cartes d'attention. Les helpers du package n'arrivent qu'a la fin pour montrer qu'ils compactent ces memes etapes sans en escamoter la logique. Les hyperparametres restent minuscules pour garder une execution rapide, avec fallback `FakeData` si CIFAR-10 n'est pas disponible localement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from vit_from_scratch.config import ViTConfig
from vit_from_scratch.data import build_dataloaders
from vit_from_scratch.model import VisionTransformer
from vit_from_scratch.patch_embedding import PatchEmbedding
from vit_from_scratch.training import evaluate, get_device, set_seed, train
from vit_from_scratch.visualization import (
    plot_class_attention,
    plot_predictions,
    plot_training_curves,
)

## Parametres miniatures

Budget de calcul volontairement bas. Avec `EPOCHS = 100` et des subsets reduits, le notebook sert a comprendre les formes et le flux de donnees.

Le nombre de patches vaut

$$N = \left(
\frac{H}{P}
\right)\left(
\frac{W}{P}
\right).$$

Sur CIFAR-10, $H = W = 32$. Avec `PATCH_SIZE = 4`, on obtient $N = 8 \times 8 = 64$ patches.

In [ ]:
set_seed(7)

DATA_DIR = Path("../data")
REQUESTED_DATASET = "cifar10"
BATCH_SIZE = 8
TRAIN_SUBSET_SIZE = 96
VAL_SUBSET_SIZE = 32
IMAGE_SIZE = 32
PATCH_SIZE = 4
EMBED_DIM = 64
NUM_CLASSES = 10
DEPTH = 2
NUM_HEADS = 4
MLP_RATIO = 2.0
EPOCHS = 100
DEVICE = get_device("auto")

print({
    "device": str(DEVICE),
    "requested_dataset": REQUESTED_DATASET,
    "batch_size": BATCH_SIZE,
    "train_subset": TRAIN_SUBSET_SIZE,
    "val_subset": VAL_SUBSET_SIZE,
})

## Classes et normalisation CIFAR-10

CIFAR-10 contient dix classes. L'entree RGB est normalisee canal par canal :

$$x_{norm}^{(c)} = 
\frac{x^{(c)} - \mu_c}{\sigma_c}.$$

Centrer les canaux rend l'optimisation plus stable.

In [ ]:
CIFAR10_CLASSES = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

print("classes:", CIFAR10_CLASSES)
print("mean:", CIFAR10_MEAN)
print("std:", CIFAR10_STD)

### Pourquoi augmenter les donnees ?

CIFAR-10 contient 50 000 images d'entrainement. Un Transformer sans inductive bias spatiale (contrairement a un CNN) est plus susceptible de memoriser les exemples qu'il voit. L'augmentation cree de la variete artificielle : une meme image recadree differemment ou retournee horizontalement represente le meme objet, mais oblige le modele a apprendre une representation invariante plutot qu'un patron pixel-exact. Sans augmentation sur un dataset aussi petit, l'ecart train/validation s'elargit rapidement, symptome classique de sur-apprentissage.

## Transforms train et validation

Cote train, on ajoute des augmentations modestes (crop aleatoire, flip horizontal). Cote validation, conversion en tenseur et normalisation uniquement.

$$T_{train} = Normalize \circ ToTensor \circ Flip \circ Crop,$$
$$T_{val} = Normalize \circ ToTensor.$$

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

train_transform, val_transform

### Pourquoi le fallback FakeData ?

CIFAR-10 n'est pas toujours disponible localement : premiere execution sur une nouvelle machine, environnement sans acces reseau, CI/CD sans stockage persistant. Plutot que d'imposer un telechargement, on bascule sur `FakeData` qui genere des tenseurs aleatoires aux memes dimensions (`3 x 32 x 32`, 10 classes). Le reste du pipeline (formes, interfaces, indices de labels) est strictement identique. Cela garantit que le notebook est executabledans n'importe quel contexte, meme si l'intuition semantique des images n'est plus presente.

## Chargement CIFAR-10 avec fallback FakeData

1. Essayer CIFAR-10 sans forcer de telechargement.
2. Si absent, basculer sur `FakeData`.
3. Conserver les memes dimensions `3 x 32 x 32` et le meme nombre de classes.

`FakeData` ne remplace pas de vraies images pour l'intuition semantique, mais suffit pour tester les interfaces tenseurs.

In [ ]:
def load_dataset_with_fallback(train: bool, transform):
    try:
        dataset = datasets.CIFAR10(
            root=DATA_DIR,
            train=train,
            transform=transform,
            download=False,
        )
        dataset_name = "cifar10"
    except Exception as error:
        size = 50_000 if train else 10_000
        offset = 0 if train else 1_000_000
        dataset = datasets.FakeData(
            size=size,
            image_size=(3, 32, 32),
            num_classes=10,
            transform=transform,
            random_offset=offset,
        )
        dataset_name = "fake"
        print(f"CIFAR-10 indisponible, fallback FakeData ({error})")
    return dataset_name, dataset

train_dataset_name, train_dataset_full = load_dataset_with_fallback(True, train_transform)
val_dataset_name, val_dataset_full = load_dataset_with_fallback(False, val_transform)
ACTIVE_DATASET = "cifar10" if train_dataset_name == "cifar10" and val_dataset_name == "cifar10" else "fake"

print({
    "train_dataset": train_dataset_name,
    "val_dataset": val_dataset_name,
    "active_dataset": ACTIVE_DATASET,
    "train_size_full": len(train_dataset_full),
    "val_size_full": len(val_dataset_full),
})

### Pourquoi sous-echantillonner ?

Entrainer sur 50 000 images avec un Transformer, meme minuscule, prendrait plusieurs minutes par epoque sur CPU. L'objectif de ce notebook est pedagogique : verifier que les formes sont correctes, que le flux de donnees fonctionne, que la boucle d'entrainement tourne sans erreur. Pour cela, 96 exemples suffisent largement. Les formes des tenseurs, la logique de batching, la passe avant et le calcul de la perte sont rigoureusement identiques a ceux d'un entrainement complet, seul le volume de donnees differe. Quand on passe a un vrai entrainement, on retire simplement les appels a `deterministic_subset`.

## Sous-echantillonnage deterministe

On fige un generateur pseudo-aleatoire et on tire un petit sous-ensemble d'indices. Si $\pi$ est une permutation pseudo-aleatoire de $\{0, \dots, n-1\}$, on prend

$$S_k = \{\pi_0, \pi_1, \dots, \pi_{k-1}\}.$$

Le contenu du subset reste identique d'une execution a l'autre tant que la seed ne change pas.

In [ ]:
def deterministic_subset(dataset, subset_size: int, seed: int):
    if subset_size >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:subset_size].tolist()
    return Subset(dataset, indices)

train_dataset = deterministic_subset(train_dataset_full, TRAIN_SUBSET_SIZE, seed=7)
val_dataset = deterministic_subset(val_dataset_full, VAL_SUBSET_SIZE, seed=8)

print("train subset:", len(train_dataset))
print("val subset:", len(val_dataset))

### Du Dataset au DataLoader : pourquoi cette distinction ?

Un `Dataset` donne un acces element par element via `__getitem__` : on peut demander l'image numero 42, mais rien de plus. Le `DataLoader` ajoute trois services essentiels : il regroupe les elements en mini-lots (batching), il melange l'ordre a chaque epoque pour eviter que le modele apprenne l'ordre des exemples (shuffling), et il peut prefetcher des donnees en arriere-plan pour ne pas laisser le GPU en attente (parallelisme via `num_workers`). Cette separation nette entre "ce que contient le dataset" et "comment on l'itere" est une conception deliberee de PyTorch, elle facilite le remplacement du dataset sans toucher a la boucle d'entrainement.

## DataLoader manuel

Un `Dataset` donne un acces element par element. Le `DataLoader` ajoute le batching, le melange et la construction des mini-lots.

Chaque image $X \in \mathbb{R}^{C \times H \times W}$ ; un mini-batch forme un tenseur :

$$X_{batch} \in \mathbb{R}^{B \times C \times H \times W}$$

avec un vecteur d'etiquettes $y \in \{0, \dots, K-1\}^B$.

In [ ]:
train_loader_manual = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader_manual = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

len(train_loader_manual), len(val_loader_manual)

## Inspection d'un batch

Avant d'entrainer, on verifie les formes. C'est la qu'on detecte la plupart des erreurs de pipeline : mauvais ordre de dimensions, labels hors plage, normalisation oubliee.

In [ ]:
train_images, train_labels = next(iter(train_loader_manual))
val_images, val_labels = next(iter(val_loader_manual))

print("train batch images:", tuple(train_images.shape))
print("train batch labels:", tuple(train_labels.shape))
print("label ids:", train_labels.tolist())
print("image dtype:", train_images.dtype)
print("pixel range after normalization:", float(train_images.min()), float(train_images.max()))

## Desnormaliser pour visualiser

La normalisation aide le modele mais rend l'image illisible. On inverse canal par canal :

$$x^{(c)} = x_{norm}^{(c)} \sigma_c + \mu_c.$$

In [ ]:
def manual_unnormalize_image(image, mean, std):
    mean_tensor = torch.tensor(mean, dtype=image.dtype, device=image.device).view(-1, 1, 1)
    std_tensor = torch.tensor(std, dtype=image.dtype, device=image.device).view(-1, 1, 1)
    return image * std_tensor + mean_tensor

example_image = manual_unnormalize_image(train_images[0], CIFAR10_MEAN, CIFAR10_STD).clamp(0.0, 1.0)
plt.figure(figsize=(3, 3))
plt.imshow(example_image.permute(1, 2, 0))
plt.title(CIFAR10_CLASSES[int(train_labels[0])])
plt.axis("off")

## Tracer manuellement la grille de patches

Le ViT traite une sequence de patches, pas des pixels individuels. Visualiser la grille aide a comprendre la resolution effective de l'attention.

In [ ]:
def manual_make_patch_grid(image, patch_size: int):
    _, height, width = image.shape
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image.permute(1, 2, 0).clamp(0.0, 1.0))
    for y in range(patch_size, height, patch_size):
        ax.axhline(y - 0.5, color="white", linewidth=0.8, alpha=0.9)
    for x in range(patch_size, width, patch_size):
        ax.axvline(x - 0.5, color="white", linewidth=0.8, alpha=0.9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Patch grid ({patch_size}x{patch_size})")
    fig.tight_layout()
    return fig

manual_make_patch_grid(example_image, PATCH_SIZE)

## Extraction explicite avec `nn.Unfold`

`PatchEmbedding` fait ce travail via une convolution stridee, mais on peut d'abord regarder l'operation avec `nn.Unfold`, qui decoupe l'image en colonnes de patches aplatis.

Pour une image unique, la sortie a la forme

$$[1, C P^2, N],$$

ou, apres transposition,

$$[1, N, C P^2].$$

In [ ]:
unfold = nn.Unfold(kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
example_batch = train_images[:1]
flat_patches = unfold(example_batch)
flat_patches = flat_patches.transpose(1, 2)

print("flat patches shape:", tuple(flat_patches.shape))
print("patch vector dim:", flat_patches.shape[-1])
print("num patches:", flat_patches.shape[1])

## `PatchEmbedding`: projection vers l'espace latent

Chaque patch aplati appartient a $\mathbb{R}^{C P^2}$. `PatchEmbedding` apprend une projection lineaire vers $\mathbb{R}^D$ :

$$e_i = W_p x_i + b_p, \quad W_p \in \mathbb{R}^{D \times (C P^2)}.$$

Dans le package, cette projection est realisee par une convolution dont le noyau et le stride valent `patch_size`.

In [ ]:
config = ViTConfig(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=3,
    num_classes=NUM_CLASSES,
    embed_dim=EMBED_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    mlp_ratio=MLP_RATIO,
    dropout=0.1,
    attention_dropout=0.1,
    position_embedding="learned",
)

patch_embed = PatchEmbedding(config)
patch_tokens = patch_embed(example_batch)
print("patch token shape:", tuple(patch_tokens.shape))

## `class token` et encodage positionnel

Les patches seuls n'ont ni token global ni information d'ordre. On prepend un vecteur appris `cls`, puis on ajoute un embedding positionnel.

Si `E` est la matrice des patch embeddings, la sequence d'entree du Transformer devient

$$Z_0 = [x_{cls}; E] + E_{pos}.$$

Le premier terme resume l'image au fil de l'attention ; le second preserve la structure spatiale perdue lors de l'aplatissement.

In [ ]:
model = VisionTransformer(config)

with torch.no_grad():
    patch_tokens = model.patch_embed(example_batch) # (B, num_patches, D)
    cls_token = model.cls_token.expand(example_batch.shape[0], -1, -1) # (B, 1, D)
    tokens_without_pos = torch.cat([cls_token, patch_tokens], dim=1) # (B, num_patches + 1, D)
    if model.position_embedding is not None:
        tokens_with_pos = model.position_embedding(tokens_without_pos.clone()) # (B, num_patches + 1, D)
    else:
        tokens_with_pos = tokens_without_pos # (B, num_patches + 1, D)

print("tokens without pos:", tuple(tokens_without_pos.shape))
print("tokens with pos:", tuple(tokens_with_pos.shape))
print("num tokens including cls:", tokens_with_pos.shape[1])

## Ce que compactent les helpers du package

Tout ce qui precede a ete reconstruit a la main. Les helpers du package reprennent les memes ingredients dans une interface plus courte pour les experiences repetables.

On les appelle maintenant : `build_dataloaders` pour la preparation, `train` pour la boucle multi-epoques, `evaluate` pour la validation et `forward_with_attention` pour visualiser le `class token`.

In [ ]:
package_dataloaders = build_dataloaders(
    dataset=ACTIVE_DATASET,
    data_dir=str(DATA_DIR),
    batch_size=BATCH_SIZE,
    num_workers=0,
    max_train_samples=TRAIN_SUBSET_SIZE,
    max_val_samples=VAL_SUBSET_SIZE,
    seed=7,
    download=False,
)

package_dataloaders.class_names

### Toutes les pieces sont en place

C'est ici que l'ensemble du pipeline se noue : les transforms definissent comment chaque image est preparee, le DataLoader fournit les mini-lots, le modele transforme chaque lot en logits, la fonction de perte mesure l'ecart avec les vraies etiquettes, et l'optimiseur ajuste les poids. La boucle d'entrainement ne fait que repeter ce cycle. On limite ici a une seule epoque, suffisante pour verifier que tout s'enchaine sans erreur et que les metriques sont calculees correctement, sans pretendre converger vers une solution utile.

## Entrainement minuscule

On utilise cross-entropy plutot que MSE parce que la sortie est une distribution de probabilites sur les classes. MSE traiterait les logits comme des valeurs continues, ignorant leur structure categorielle.

$$\mathcal{L}_{cls} = -\sum_{k=1}^{K} y_k \log p_k,$$

avec $p = softmax(f_{\theta}(x))$. Une seule epoque suffit ici pour produire une chaine complete : batchs, tokens, optimisation, evaluation, prediction, attention.

In [ ]:
model = VisionTransformer(config).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

history = train(
    model=model,
    train_loader=package_dataloaders.train_loader,
    val_loader=package_dataloaders.val_loader,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS,
)
val_metrics = evaluate(
    model=model,
    dataloader=package_dataloaders.val_loader,
    device=DEVICE,
)

print("history keys:", list(history))
print("val metrics:", val_metrics)

In [ ]:
plot_training_curves(history)
plt.show()

preview_images, preview_labels = next(iter(package_dataloaders.val_loader))
with torch.no_grad():
    preview_logits = model(preview_images.to(DEVICE)).cpu()
preview_display = torch.stack([
    manual_unnormalize_image(image, CIFAR10_MEAN, CIFAR10_STD).clamp(0.0, 1.0)
    for image in preview_images[:8]
])
plot_predictions(
    images=preview_display,
    labels=preview_labels[:8],
    logits=preview_logits[:8],
    class_names=package_dataloaders.class_names,
    max_images=8,
)
plt.show()

## Attention du `class token`

### Que montrent les cartes d'attention ?

Le `class token` (position 0 dans la sequence) est le vecteur que le modele utilise pour produire la prediction finale. A chaque couche d'attention, ce token interroge tous les autres tokens (y compris lui-meme) et pondere leur contribution par un score d'attention. La ligne 0 de la matrice d'attention $A \in \mathbb{R}^{H \times N \times N}$ correspond exactement a ces poids : elle indique quels patches spatiaux le `class token` juge les plus informatifs.

Pourquoi la ligne 0 ? Parce que le `class token` est prepend a la sequence a la position 0. Ses scores d'attention vers les positions $1, \dots, N$ representent sa "lecture" de chaque patch de l'image. En remappant ces $N$ scores sur la grille spatiale $\sqrt{N} \times \sqrt{N}$, on obtient une carte de saillance interpretable.

Dans un modele bien entraine, ces cartes mettent souvent en evidence les contours des objets, les textures caracteristiques ou les regions semantiquement pertinentes pour la classe predite. Ici, apres une quelques epoques sur un tiny subset, les cartes sont encore largement bruit, mais la machinerie est identique a celle d'un ViT entraine sur ImageNet.

Le mode `forward_with_attention` expose les matrices $A \in \mathbb{R}^{H \times N \times N}$ de chaque bloc. La ligne du `class token` indique quels patches contribuent le plus a sa representation globale.

In [ ]:
attention_input = preview_images[:1].to(DEVICE)
with torch.no_grad():
    attention_logits, attention_maps = model.forward_with_attention(attention_input)

attention_display = manual_unnormalize_image(preview_images[0], CIFAR10_MEAN, CIFAR10_STD).clamp(0.0, 1.0)
predicted_index = int(attention_logits.argmax(dim=1).item())

plot_class_attention(
    attention_maps=attention_maps,
    image=attention_display,
    patch_size=PATCH_SIZE,
    layer=-1,
    head="mean",
)
plt.suptitle(f"Prediction: {package_dataloaders.class_names[predicted_index]}", y=1.02)
plt.show()

## Conclusion

### Ce que ce notebook a couvert

En parcourant ce notebook de bout en bout, on a traverse l'ensemble de la chaine d'un Vision Transformer :

1. **Transforms**, normalisation canal par canal et augmentations geometriques pour regulariser l'apprentissage sur un petit dataset.
2. **Dataset et fallback**, chargement de CIFAR-10 avec repli sur `FakeData` pour garantir l'executabilite.
3. **Sous-echantillonnage deterministe**, reduction du volume pour une execution rapide, sans changer la logique du pipeline.
4. **DataLoader**, batching, melange et parallelisme ; separation nette entre "ce que l'on charge" et "comment on itere".
5. **Inspection d'un batch**, verification des formes, des types et de la plage de valeurs avant tout entrainement.
6. **Extraction des patches et projection**, decoupage de l'image en une sequence de vecteurs via `nn.Unfold` puis `PatchEmbedding`.
7. **Construction de la sequence**, ajout du `class token` et de l'encodage positionnel appris.
8. **Entrainement**, boucle complete avec cross-entropy, AdamW, et evaluation apres chaque epoque.
9. **Evaluation et predictions**, metriques de validation et visualisation des predictions sur un batch.
10. **Cartes d'attention**, visualisation de ce que le `class token` regarde dans l'image.

### Ce qui differe dans un vrai entrainement

Ce notebook est volontairement minimal. Un entrainement serieux sur CIFAR-10 ou ImageNet ajoute plusieurs elements absents ici :

- **Plus d'epoques** (typiquement 90–300) avec un learning rate schedule (warmup cosine ou step decay).
- **Augmentations plus agressives** : mixup, cutmix, RandAugment, label smoothing.
- **Checkpointing** : sauvegarde du meilleur modele au fil des epoques.
- **Gradient clipping** pour stabiliser l'entrainement des Transformers.
- **Pre-entrainement sur un plus grand dataset** (ImageNet-21k ou JFT) suivi d'un fine-tuning, c'est la recette originale du papier ViT.

### Prochaines etapes

- `training_methods.ipynb`, comparaison de differents objectifs d'entrainement (supervision, auto-supervision, contrastif).
- `experiment_results.ipynb`, analyse de runs reels : courbes de convergence, ablations sur la profondeur et la taille des patches, comparaison avec un ResNet.